# Table Models for Observability Data

**Module 4: Table Models for Observability Data** · [Quiz](quiz04_observability_table_models.ipynb) · [Course README](../../README.md)

Every Doris table has a data model, set by the key clause of `CREATE TABLE`. The model decides what Doris does with rows that have the same values in the key columns:

| Model | Rows with the same key | Use in observability |
| --- | --- | --- |
| `DUPLICATE KEY` | All are kept. The key columns only set the sort order. | Raw logs, spans, and metric points |
| `AGGREGATE KEY` | Combined into one row. Each value column names its aggregate function, such as `SUM`. | Pre-computed counts, such as errors per service and day |
| `UNIQUE KEY` | Only the row loaded last is kept. | Current state, such as one alert per problem and day |

In this Lab, you check which models the exporter chose for the telemetry tables, build an AGGREGATE table that counts logs and errors per service and day, and build a UNIQUE table that keeps one alert per problem and day. The examples read the telemetry that Module 1 (the demo) and Module 2 (the FE log) loaded, mostly the logs in `otel_logs`. The new tables are created in the `otel` database, and every cell can be rerun.

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

## 1. Check the logs in Doris

The examples count log rows and error rows per service in `otel_logs`. Each log row has two severity fields. `severity_text` is the label that the source wrote: `ERROR` in the FE log, `Error` in .NET services such as `accounting`, and nothing in the proxy's access log. `severity_number` puts every source on the same OpenTelemetry scale: 9–12 is INFO, 13–16 WARN, 17–20 ERROR, and 21–24 FATAL. The examples therefore treat a row as an error when its `severity_number` is 17 or more. Count the rows by service:

In [ ]:
log_services = lab.wait_for_rows(
    """
    SELECT service_name,
           COUNT(*) AS log_rows,
           SUM(IF(severity_number >= 17, 1, 0)) AS error_rows,
           MAX(timestamp) AS latest_log
    FROM otel_logs
    GROUP BY service_name
    ORDER BY log_rows DESC
    """,
    title="Log rows and error rows by service in otel_logs",
    hint="otel_logs has no rows. Start the DOG stack and the demo with the cells in Module 1, "
         "and load the FE log with section 1 of Module 2. Then rerun this cell.",
)

**Expected result**

One row per service, about 11 once the demo has run for a few minutes. `frontend-proxy` and `doris-fe` have the most rows; `currency`, `cart`, `kafka`, and the other demo services follow. Most services have no error rows. `doris-fe` has a few, which the FE logged while it started, and a demo service such as `accounting` may have one. Section 4 builds alerts from these error rows: if `doris-fe` is missing, load the FE log with section 1 of Module 2, then rerun this cell.

## 2. DUPLICATE: keep every event

The Doris exporter created the telemetry tables when it started in Module 1. `information_schema.columns` marks each key column with its table's model: `DUP` for DUPLICATE, `AGG` for AGGREGATE, and `UNI` for UNIQUE. List the models and key columns of these tables:

In [ ]:
lab.sql("""
SELECT TABLE_NAME AS table_name,
       MAX(COLUMN_KEY) AS model,
       GROUP_CONCAT(COLUMN_NAME ORDER BY ORDINAL_POSITION SEPARATOR ', ') AS key_columns
FROM information_schema.columns
WHERE TABLE_SCHEMA = 'otel' AND TABLE_NAME LIKE 'otel%' AND COLUMN_KEY <> ''
GROUP BY TABLE_NAME
ORDER BY table_name
""", title="Models and key columns of the telemetry tables");

**Expected result**

Eight tables. Seven are `DUP`: `otel_logs`, `otel_traces`, and the five `otel_metrics_*` tables, keyed on `timestamp` and `service_name` (in either order). `otel_traces_graph` is `UNI`; section 4 explains why. Tables that later modules create also start with `otel_`, so they are listed too once you have run those Labs.

A log line, a span, and a metric point are each a separate event, so the raw tables use DUPLICATE: Doris keeps every row it loads. The key columns only set the sort order, which helps queries that filter on time and service. DUPLICATE is also the model that `CREATE TABLE` uses when it has no key clause.

The key of `otel_logs` is `timestamp` and `service_name`. Look for keys that several log rows share:

In [ ]:
lab.sql("""
SELECT timestamp, service_name,
       COUNT(*) AS rows_with_this_key,
       COUNT(DISTINCT body) AS different_messages
FROM otel_logs
GROUP BY timestamp, service_name
HAVING COUNT(*) > 1
ORDER BY rows_with_this_key DESC, timestamp
LIMIT 5
""", title="Keys that several rows in otel_logs share");

**Expected result**

Five keys, each shared by several rows, usually from `doris-fe`: the FE log records the time to the millisecond, and the FE often writes several lines in the same millisecond. `different_messages` equals `rows_with_this_key`, so these rows are different events, and a table that kept one row per key would lose all but one of them. If you started the log collector of Module 2 more than once, it loaded the FE log again, and `different_messages` is smaller than `rows_with_this_key`: DUPLICATE keeps exact copies too, so it cannot protect a table from a repeated load.

## 3. AGGREGATE: count logs and errors per service and day

A dashboard of errors per service and day would count them in `otel_logs` on every refresh, reading every log row. An AGGREGATE table keeps the counts instead, one row per service and day.

In the next table, the key columns are the reporting grain: `log_date` and `service_name`. The value columns `log_count` and `error_count` declare `SUM`: when a row arrives whose key the table already has, Doris adds its values to the existing row. Doris combines such rows while it loads them, merges the rest in the background (compaction), and combines anything left at query time, so a query always sees one row per key.

The key holds the day, not the timestamp. With `timestamp` in the key, only rows from the same microsecond would be combined, and the table would be almost as large as `otel_logs`. `log_date` is the UTC day, because the exporter stores UTC times. `DISTRIBUTED BY HASH(service_name)` sends all rows of a service to the same tablet, where loads and compaction can combine them; one bucket is enough for a table this small. Module 5 explains how to choose partitions and buckets.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS daily_log_counts (
    log_date     DATE,
    service_name VARCHAR(200),
    log_count    BIGINT SUM,
    error_count  BIGINT SUM
)
AGGREGATE KEY(log_date, service_name)
DISTRIBUTED BY HASH(service_name) BUCKETS 1
""", title="Create the AGGREGATE table daily_log_counts");

**Expected result**

A card says that the statement succeeded. `IF NOT EXISTS` leaves an existing table unchanged, so you can rerun the cell.

### Load the logs in two batches

A load job usually runs on a schedule and adds the rows of the period that has just ended. The next cell loads `otel_logs` into `daily_log_counts` in two such batches: the second batch is 30 seconds of logs, and the first batch is everything before it. Rows reach Doris some seconds after they are written, so the cell leaves out the last 30 seconds before the newest log row.

Each log row becomes one row with `log_count = 1`, and `error_count = 1` for an error or `0` otherwise. Before you run the cell, predict: after the second batch, does the table have more rows than after the first? What happens to the `log_count` of a service that logged in both batches?

In [ ]:
bounds = lab.sql("""
SELECT MAX(timestamp) - INTERVAL 60 SECOND AS second_batch_start,
       MAX(timestamp) - INTERVAL 30 SECOND AS second_batch_end
FROM otel_logs
""", title="Time range of the second batch")
second_batch_start = bounds["second_batch_start"][0]
second_batch_end = bounds["second_batch_end"][0]

lab.execute("TRUNCATE TABLE daily_log_counts", title="Empty the table, so that the cell can be rerun")
lab.execute(f"""
INSERT INTO daily_log_counts (log_date, service_name, log_count, error_count)
SELECT DATE(timestamp), service_name, 1, IF(severity_number >= 17, 1, 0)
FROM otel_logs
WHERE timestamp < '{second_batch_start}'
""", title="First batch: the log rows before the second batch")
lab.sql("SELECT * FROM daily_log_counts ORDER BY log_count DESC", title="daily_log_counts after the first batch")

lab.execute(f"""
INSERT INTO daily_log_counts (log_date, service_name, log_count, error_count)
SELECT DATE(timestamp), service_name, 1, IF(severity_number >= 17, 1, 0)
FROM otel_logs
WHERE timestamp >= '{second_batch_start}' AND timestamp < '{second_batch_end}'
""", title="Second batch: 30 seconds of log rows")
lab.sql("SELECT * FROM daily_log_counts ORDER BY log_count DESC", title="daily_log_counts after the second batch");

**Expected result**

The first batch reports one affected row for each log row it loaded, hundreds or more, yet `daily_log_counts` has only one row per service and day. The second batch loads 30 seconds of logs, hundreds or thousands of rows. Afterwards, the table still has one row per service and day: a service that logged in both batches, such as `doris-fe` or `frontend-proxy`, keeps its row, and its `log_count` has grown by its rows in the second batch. This is the `SUM` merge: loads of 4 and then 7 for one key leave one row with 11. A service that logged only in the second batch gets a new row.

If the first batch loads no rows, the logs in Doris cover less than a minute: wait a minute, then rerun the cell.

An AGGREGATE table cannot tell new rows from a repeated load. Had the second batch been loaded twice, its rows would have been added twice, and the counts would be too high. A load job must load every source row exactly once; this cell starts with `TRUNCATE TABLE`, so you can rerun it.

Check the counts against the raw rows. The next query computes them from `otel_logs`, up to the end of the second batch:

In [ ]:
lab.sql(f"""
SELECT DATE(timestamp) AS log_date, service_name,
       COUNT(*) AS log_count,
       SUM(IF(severity_number >= 17, 1, 0)) AS error_count
FROM otel_logs
WHERE timestamp < '{second_batch_end}'
GROUP BY DATE(timestamp), service_name
ORDER BY log_count DESC
""", title="The same counts computed from otel_logs");

**Expected result**

The same rows and numbers as `daily_log_counts` after the second batch. This query reads every log row up to `second_batch_end`; `daily_log_counts` gives the same answer from one row per service and day. The price is that the table answers only questions at its grain: it cannot say which errors a service had, or how many it had in one hour.

If the numbers differ, some rows of that period reached Doris after the cell above loaded it: rerun both cells.

## 4. UNIQUE: one alert per problem and day

Suppose that an alerting job reads the error rows every minute and raises an alert for each. When a problem repeats, such as `Failed to build distribute plans` in the FE log, one alert per error row is an alert storm. A UNIQUE table keeps one row per key, so with the day and an alarm identity as the key, each problem raises one alert per day, however often its error repeats.

The alarm identity must be the same for every occurrence of a problem and differ between problems. Error messages contain parts that change with every occurrence, such as timestamps, thread numbers, and IDs. The query below takes the first line of the message (the rest can be a stack trace), replaces every number with `0`, and hashes the result together with the service name: `alarm_hash`. See how it groups the error rows:

In [ ]:
alarms = lab.wait_for_rows(
    r"""
    SELECT DATE(timestamp) AS alarm_date,
           service_name,
           MD5(CONCAT_WS('|', service_name, REGEXP_REPLACE(SUBSTRING_INDEX(body, '\n', 1), '[0-9]+', '0'))) AS alarm_hash,
           COUNT(*) AS error_rows,
           LEFT(MIN(REGEXP_REPLACE(SUBSTRING_INDEX(body, '\n', 1), '[0-9]+', '0')), 120) AS normalized_message
    FROM otel_logs
    WHERE severity_number >= 17
    GROUP BY alarm_date, service_name, alarm_hash
    ORDER BY error_rows DESC, alarm_hash
    """,
    title="Error rows by alarm identity",
    hint="otel_logs has no error rows. Load the FE log with section 1 of Module 2: "
         "the FE logs a few errors while it starts. Then rerun this cell.",
    timeout_seconds=10,
)

**Expected result**

One row per alarm and day, usually four or five, most from `doris-fe`. The rows of one problem share an identity: the `Failed to build distribute plans` errors, for example, have the same `normalized_message` and `alarm_hash`, although their timestamps and thread numbers differ, and `error_rows` shows how many there are. Different messages get different identities, even when they look alike, such as the three `log4j:WARN` lines that the FE writes when it starts.

Create the alert table. Its key is the day and the alarm identity:

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS error_alerts (
    alarm_date   DATE,
    alarm_hash   VARCHAR(32),
    service_name VARCHAR(200),
    status       VARCHAR(16),
    message      STRING
)
UNIQUE KEY(alarm_date, alarm_hash)
DISTRIBUTED BY HASH(alarm_hash) BUCKETS 1
""", title="Create the UNIQUE table error_alerts");

**Expected result**

A card says that the statement succeeded. A UNIQUE table must be distributed by a hash of key columns, so that all rows with one key reach the same tablet. Doris 4.x creates UNIQUE tables as Merge-on-Write (MoW) by default: when a row arrives whose key the table already has, Doris marks the old row as deleted while it loads the new one, so queries read only the newest row.

Raise an open alert for every error row, as the alerting job would:

In [ ]:
RAISE_ALERTS = r"""
INSERT INTO error_alerts (alarm_date, alarm_hash, service_name, status, message)
SELECT DATE(timestamp),
       MD5(CONCAT_WS('|', service_name, REGEXP_REPLACE(SUBSTRING_INDEX(body, '\n', 1), '[0-9]+', '0'))),
       service_name,
       'open',
       SUBSTRING_INDEX(body, '\n', 1)
FROM otel_logs
WHERE severity_number >= 17
"""
lab.execute("TRUNCATE TABLE error_alerts", title="Empty the table, so that the cell can be rerun")
lab.execute(RAISE_ALERTS, title="Raise an open alert for every error row")
lab.sql("SELECT * FROM error_alerts ORDER BY service_name, alarm_hash", title="error_alerts");

**Expected result**

The insert reports one affected row per error row, for example 16, yet `error_alerts` has one row per alarm and day, as many as the previous query returned: rows with the same key replaced each other. Each row keeps the message of one occurrence, the one that Doris loaded last.

One row does not mean one error. The `Failed to build distribute plans` alarm has one row in `error_alerts` and many error rows in `otel_logs`. The table keeps the current state, not a count: to know how often a problem occurred, count its rows in `otel_logs`, as the previous query did, or keep a count in an AGGREGATE table.

A minute later, the alerting job runs again and reads the same error rows. Load them once more:

In [ ]:
lab.execute(RAISE_ALERTS, title="Raise the alerts again from the same error rows")
lab.sql("SELECT COUNT(*) AS alerts FROM error_alerts", title="Alerts after the second load");

**Expected result**

The insert loads the same number of rows again, and `alerts` has not changed. Repeated loads replace rows instead of adding them, so a UNIQUE table can be loaded again safely. A repeated load of `daily_log_counts` in section 3 would have counted every row twice.

When a problem is fixed, the alerting job writes its alarm again, with the status `resolved`. Resolve the alarm with the most error rows:

In [ ]:
alarm_hash = alarms["alarm_hash"][0]
lab.execute(f"""
INSERT INTO error_alerts (alarm_date, alarm_hash, service_name, status, message)
SELECT alarm_date, alarm_hash, service_name, 'resolved', message
FROM error_alerts
WHERE alarm_hash = '{alarm_hash}'
""", title="Write the alarm again with the status resolved")
lab.sql(f"SELECT * FROM error_alerts WHERE alarm_hash = '{alarm_hash}'", title="The resolved alarm")
lab.sql("SELECT status, COUNT(*) AS alerts FROM error_alerts GROUP BY status ORDER BY status", title="Alerts by status");

**Expected result**

The insert writes one row, and the alarm still has one row, now with `status = resolved`. `Alerts by status` adds up to the same number of alerts as before, one of them resolved: a later load with the same key replaced the open row.

If you now rerun the cell that raises the alerts again, this alarm is `open` again, because that load is later still: UNIQUE keeps the row loaded last. When loads can arrive out of order, give the table a sequence column (the `function_column.sequence_col` property): Doris then keeps the row with the largest value in that column, such as the newest occurrence time, whatever the load order.

### The exporter's UNIQUE table

`otel_traces_graph`, the UNIQUE table from section 2, counts the calls between services per minute. A job that the exporter created, `otel_traces_graph_job`, fills it every 10 minutes: each run counts the spans in `otel_traces` from the start of the minute 10 minutes earlier up to the current time. Each run therefore counts again the minute in which the previous run started, which the previous run saw only in part, and writes rows with the same keys. UNIQUE replaces the earlier, partial counts with the new ones. With AGGREGATE `SUM`, the two counts would be added; with DUPLICATE, the minute would appear twice.

Look at the job with `jobs('type'='insert')`, which lists the scheduled insert jobs, and at the calls from `frontend-proxy` to `frontend` in the newest minutes:

In [ ]:
lab.sql("""
SELECT Name, RecurringStrategy, Status, SucceedTaskCount, LastTaskSuccessTime
FROM jobs('type'='insert')
WHERE Name LIKE '%otel_traces_graph_job'
""", title="The job that fills otel_traces_graph")
lab.sql("""
SELECT timestamp, caller_service_name, callee_service_name, count, error_count
FROM otel_traces_graph
WHERE caller_service_name = 'frontend-proxy' AND callee_service_name = 'frontend'
ORDER BY timestamp DESC
LIMIT 5
""", title="Calls from frontend-proxy to frontend per minute");

**Expected result**

The job runs `EVERY 10 MINUTE`, and `SucceedTaskCount` says how often it has run. Most minutes have a similar number of calls, about 1,450 while the demo runs. The newest minutes can have fewer calls, or no row yet: the last run counted them before all of their spans had reached Doris. Note the minute of `LastTaskSuccessTime` and rerun this cell after the next run, 10 minutes later: that minute now has its complete count, in one row.

If `SucceedTaskCount` is 0 and the second table is empty, the job has not run yet. Whenever a collector that exports traces starts, such as the DOG Collector in Module 1 or the trace collector in Module 2, the exporter drops the job and creates it again, and the new job first runs 10 minutes later. Rerun this cell then.

## 5. Independent exercise

Build an AGGREGATE table of spans: `daily_span_counts`, with the key `span_date` and `service_name`, and the `SUM` columns `span_count` and `error_span_count`. A span is an error when its `status_code` is `STATUS_CODE_ERROR`. Load it from `otel_traces` up to a fixed time, 30 seconds before the newest span, then compare it with a `GROUP BY` query on `otel_traces` with the same time limit. Expect one row per service and day, the same numbers in both results, and error spans in a few services at most.

Write and run your code in the next cell, then expand the reference solution.

<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
span_limit = lab.sql(
    "SELECT MAX(timestamp) - INTERVAL 30 SECOND AS span_limit FROM otel_traces",
    title="Load the spans before this time",
)["span_limit"][0]

lab.execute("""
CREATE TABLE IF NOT EXISTS daily_span_counts (
    span_date        DATE,
    service_name     VARCHAR(200),
    span_count       BIGINT SUM,
    error_span_count BIGINT SUM
)
AGGREGATE KEY(span_date, service_name)
DISTRIBUTED BY HASH(service_name) BUCKETS 1
""", title="Create the AGGREGATE table daily_span_counts")
lab.execute("TRUNCATE TABLE daily_span_counts", title="Empty the table, so that the cell can be rerun")
lab.execute(f"""
INSERT INTO daily_span_counts (span_date, service_name, span_count, error_span_count)
SELECT DATE(timestamp), service_name, 1, IF(status_code = 'STATUS_CODE_ERROR', 1, 0)
FROM otel_traces
WHERE timestamp < '{span_limit}'
""", title="Load the spans")
lab.sql("SELECT * FROM daily_span_counts ORDER BY span_count DESC", title="daily_span_counts")
lab.sql(f"""
SELECT DATE(timestamp) AS span_date, service_name,
       COUNT(*) AS span_count,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS error_span_count
FROM otel_traces
WHERE timestamp < '{span_limit}'
GROUP BY DATE(timestamp), service_name
ORDER BY span_count DESC
""", title="The same counts computed from otel_traces");
```

</details>

## Completion criteria

- You can say which model the exporter uses for raw telemetry, and why rows that share a key in `otel_logs` must all be kept.
- `daily_log_counts` has one row per service and day; the second batch added to the existing rows, and the counts match the `GROUP BY` on `otel_logs`.
- `error_alerts` has one row per alarm and day, however often its error occurred and however often the alerts were loaded; the alarm that you resolved shows `resolved`.
- You can explain why `otel_traces_graph` is a UNIQUE table.

The Lab's tables are small. Module 5 looks at the bucketing of `daily_log_counts`, so keep that table. To remove the others, run `DROP TABLE IF EXISTS error_alerts` and `DROP TABLE IF EXISTS daily_span_counts` with `lab.execute(...)` or in a MySQL client.

## Next

Open [Quiz 4: Table Models for Observability Data](quiz04_observability_table_models.ipynb), then continue to [Module 5: Partitioning and Bucketing for Observability Data](../module05-partition-bucket/lab05_partition_and_bucket.ipynb).